In [ ]:
from langchain_community.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS


DATA_PATH="data/"

def load_pdf_files(data) :
    loader = DirectoryLoader(data,glob='*.pdf',loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

documents = load_pdf_files(data=DATA_PATH)
print("LLLL",len(documents))


def create_chunks(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

text_chunks = create_chunks(documents)
print("Chunks",len(text_chunks))


LLLL 940
Chunks 8846


In [ ]:

def get_embedding_model():
    embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-l6-v2")
    return embedding_model
embedding_model = get_embedding_model()

DB_FIAAS_PATH ="vectorstore/db_fiaas"
db=FAISS.from_documents(text_chunks,embedding_model)
db.save_local(DB_FIAAS_PATH)

In [ ]:
import os 
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chains import RetrievalQA

# Hugging Face API token
HF_TOKEN = os.getenv("HF_TOKEN")
DB_FIAAS_PATH = "vectorstore/db_fiaas"

# Hugging Face Model ID
HUGGINGFACE_REPO_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# ✅ Fix: Correct way to initialize HuggingFaceEndpoint
def load_llm(huggingface_repo_id):
    llm = HuggingFaceEndpoint(
         repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    temperature=0.5,
    model_kwargs={"max_length": 512},
    huggingfacehub_api_token=HF_TOKEN
    )
    return llm

# Load the LLM
llm = load_llm(HUGGINGFACE_REPO_ID)
print("✅ Hugging Face LLM loaded successfully!")


# Step 2: Connect LLM with FAISS and Create chain
CUSTOM_PROMPT_TEMPLATE = """
Use the pieces of information provided in the context to answer the user's question.
If you don't know the answer, just say that you don't know. Don't try to make up an answer.
Don't provide anything out of the given context.

Context: {context}
Question: {question}

Start the answer directly. No small talk, please.
"""

def set_custom_prompt(custom_prompt_template):
    return PromptTemplate(template=custom_prompt_template, input_variables=["question", "context"])

# Load FAISS embedding model
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-l6-v2")

# ✅ Fix: Ensure FAISS DB exists before loading
if os.path.exists(DB_FIAAS_PATH):
    db = FAISS.load_local(DB_FIAAS_PATH, embedding_model, allow_dangerous_deserialization=True)
    print("✅ FAISS database loaded successfully!")
else:
    print("⚠️ FAISS database not found at:", DB_FIAAS_PATH)
    exit(1)

# Create QA chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,  # ✅ No need to reload LLM again
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
    chain_type_kwargs={'prompt': set_custom_prompt(CUSTOM_PROMPT_TEMPLATE)}
)

# ✅ Fix: Ensure correct input format
user_query = input("Write query here: ")

response = qa_chain.invoke(user_query)  # ✅ Fix: Pass only the query string

# ✅ Fix: Ensure response format
if isinstance(response, dict) and "result" in response:
    print("RESULT:", response["result"])
    print("SOURCE DOCUMENTS:", response.get("source_documents", []))
else:
    print("⚠️ Unexpected response format:", response)


WARNING! max_length is not default parameter.
                    max_length was transferred to model_kwargs.
                    Please make sure that max_length is what you intended.


ValidationError: 1 validation error for HuggingFaceEndpoint
  Value error, Could not authenticate with huggingface_hub. Please check your API token. [type=value_error, input_value={'repo_id': 'mistralai/Mi...stral-7B-Instruct-v0.3'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error

In [ ]:
from huggingface_hub import InferenceClient

HUGGINGFACE_TOKEN = ""

client = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.3",
    token=HUGGINGFACE_TOKEN
)

response = client.chat_completion(
    messages=[{"role": "user", "content": "Hello!"}]
)
print(response)


ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content="Hello! How can I assist you today? I'm here to help answer questions, provide information, or just chat. What's on your mind? 😊", tool_calls=None), logprobs=None)], created=1740943246, id='', model='mistralai/Mistral-7B-Instruct-v0.3', system_fingerprint='3.0.1-sha-bb9095a', usage=ChatCompletionOutputUsage(completion_tokens=35, prompt_tokens=5, total_tokens=40), object='chat.completion')
